In [1]:
import os
import sys

sys.path.append("C:\_Lib\python\clab\experiments")
import zurich_exp as zi

In [2]:
file_path = "S:\_Data\\20250620_SMMC1_cd13_tmonB7_storTeflon\zurich\laboneq_organized\qubit_params\qubit_params.py"
qubit_params_module = zi.load_qubit_params(file_path)
qubit_params_module

<module 'qubit_params' from 'S:\\_Data\\20250620_SMMC1_cd13_tmonB7_storTeflon\\zurich\\laboneq_organized\\qubit_params\\qubit_params.py'>

In [3]:
qubit_params_module.readout_pulse.length

2e-06

In [4]:
qubit_parameters = qubit_params_module.__dict__["qubit_parameters"]
qubit_parameters

{'q0': {'qb_freq': 212876765.0163832,
  'qb_pi_len': 1.0379561412587699e-07,
  'qb_pi_amp': 0.9,
  'qb_ef_freq': 104239839.73210049,
  'qb_ef_len': 3.7167577523869886e-08,
  'qb_ef_amp': 1,
  'qb_drive_dBm_range': -30,
  'qb_resolved_pi_len': 1.2438222960677161e-05,
  'qb_resolved_pi_amp': 0.3,
  'qb_resolved_pi_muted_amp': 0.1,
  'qb_ge_pulse_type': 'gaussian',
  'qb_ef_pulse_type': 'gaussian',
  'qb_resolved_pulse_type': 'const',
  'reset_delay': 0.0003,
  'cavity_reset_delay': 0.0012,
  'ro_len': 2e-06,
  'ro_freq': 92399354.55756232,
  'ro_amp': 0.9,
  'ro_delay': 0.0,
  'ro_int_delay': 0.0,
  'ro_drive_dBm_range': 10,
  'ro_acq_dBm_range': 0,
  'sb_f0g1_alice_freq': 100000000.0,
  'sb_f0g1_alice_flat_len': 3e-06,
  'sb_f0g1_alice_amp': 1,
  'sb_f1g2_alice_freq': 100000000.0,
  'sb_f1g2_alice_flat_len': 2e-06,
  'sb_f1g2_alice_amp': 0.6,
  'sb_f2g3_alice_freq': 98800000.0,
  'sb_f2g3_alice_flat_len': 2.5e-06,
  'sb_f2g3_alice_amp': 1,
  'sb_f3g4_alice_freq': 98200000.0,
  'sb_f3g4_

In [5]:
qubit_params_module.acquire_kernel

PulseFunctional(
│   function='const',
│   uid='acquire_kernel',
│   amplitude=0.9,
│   length=2e-06,
│   can_compress=False,
│   pulse_parameters=None
)


In [6]:
ge_X180 = qubit_params_module.ge_X180

In [7]:
ge_X180

PulseFunctional(
│   function='gaussian',
│   uid='ge_X180_pulse',
│   amplitude=0.9,
│   length=1.0379561412587699e-07,
│   can_compress=False,
│   pulse_parameters=None
)


In [47]:
from laboneq.simple import DeviceSetup, Session, LinearSweepParameter

import re, h5py, os, json, numpy as np


def get_next_filename(data_path, file_name, file_type="h5"):
    pattern = re.compile(rf"^(\d+)_({re.escape(file_name)})\.{re.escape(file_type)}$")
    max_index = -1
    for entry in os.scandir(data_path):
        match = pattern.match(entry.name)
        if match:
            index = int(match.group(1))
            max_index = max(max_index, index)
    return f"{max_index + 1:05d}_{file_name}.{file_type}"


def save_data(data_path, file_name, result, config=None):
    file_path = os.path.join(data_path, get_next_filename(data_path, file_name, "h5"))

    result = result.acquired_results["ac_0"]

    with h5py.File(file_path, "w") as f:
        axis_name = np.squeeze(result.axis_name)
        axis = np.squeeze(result.axis)
        
        for name, data in zip(axis_name, axis):
            f.create_dataset(name, data=data)

        f.create_dataset("avgi", data=np.real(result.data))
        f.create_dataset("avgq", data=np.imag(result.data))

        if config:
            f.attrs["config"] = json.dumps(config)

    print("File saved at", file_path)

In [53]:
device_setup = DeviceSetup.from_yaml(
    os.getcwd() + "/helper_files/default_descriptor.yaml",
    server_host="127.0.0.1",  # ip address of the LabOne dataserver used to communicate with the instruments
    server_port="8004",  # port number of the dataserver - default is 8004
    setup_name="ZI_test",  # setup name
)

serial_num = "dev12460"

qubit_params_file_path = file_path

# create and connect to session
emulate = True
session = Session(device_setup=device_setup)
session.connect(do_emulation=emulate)

exp = zi.ramsey_ge(
    device_setup=device_setup,
    serial_num=serial_num,
    qubit_params_file_path=qubit_params_file_path,
    average_exponent=13,
    time_swp=LinearSweepParameter(uid="xpts", start=0, stop=2e-6, count=40),
)
compiled_exp = session.compile(exp)
print(
    f"{compiled_exp.estimated_runtime}s estimated runtime, excluding python, communication, and near-time operations overheads"
)
result = session.run(compiled_exp)

[2025.07.02 17:42:27.295] INFO    Logging initialized from [Default inline config in laboneq.laboneq_logging] logdir is c:\_Lib\python\clab\experiments\zurich_exp\test\laboneq_output\log
[2025.07.02 17:42:27.298] INFO    VERSION: laboneq 2.51.0
[2025.07.02 17:42:27.299] INFO    Connecting to data server at 127.0.0.1:8004
[2025.07.02 17:42:27.300] INFO    Connected to Zurich Instruments LabOne Data Server version 25.04.0.628 at 127.0.0.1:8004
[2025.07.02 17:42:27.301] WARNING SHFQC/QA:dev12460: Include the device options 'SHFQC/QC6CH' in the device setup ('options' field of the 'instruments' list in the device setup descriptor, 'device_options' argument when constructing instrument objects to be added to 'DeviceSetup' instances). This will become a strict requirement in the future.
[2025.07.02 17:42:27.302] INFO    Configuring the device setup
[2025.07.02 17:42:27.303] INFO    The device setup is configured
[2025.07.02 17:42:27.311] INFO    Starting LabOne Q Compiler run...
[2025.07.02 

In [54]:
save_data('data', 'ramsey_ge_save_test', result, config={'test': 0})

File saved at data\00001_ramsey_ge_save_test.h5


In [55]:
f = h5py.File("data/00001_ramsey_ge_save_test.h5")

In [56]:
f.keys()

<KeysViewHDF5 ['avgi', 'avgq', 'par15', 'xpts']>

In [ ]:
result = session.run(compiled_exp).acquired_results["ac_0"]

[2025.07.02 17:28:14.177] WARNING SHFQC/QA:dev12460: Device output muting is enabled, but the device is not SHF+ and therefore no muting will happen. It is suggested to disable it.
[2025.07.02 17:28:14.178] WARNING SHFQC/SG:dev12460: Device output muting is enabled, but the device is not SHF+ and therefore no mutting will happen. It is suggested to disable it.
[2025.07.02 17:28:14.179] INFO    Starting near-time execution...
[2025.07.02 17:28:14.195] INFO    Estimated RT execution time: 197.68 s.
[2025.07.02 17:28:14.195] INFO    Finished near-time execution.


In [14]:
exp.get_calibration()

Calibration(
│   calibration_items={
│   │   'qb_drive': SignalCalibration(
│   │   │   amplitude=None,
│   │   │   delay_signal=None,
│   │   │   local_oscillator=Oscillator(
│   │   │   │   uid='dev12460_SG0_lo',
│   │   │   │   frequency=4000000000.0,
│   │   │   │   modulation_type=ModulationType.AUTO,
│   │   │   │   carrier_type=None
│   │   │   ),
│   │   │   voltage_offset=None,
│   │   │   mixer_calibration=None,
│   │   │   precompensation=None,
│   │   │   oscillator=Oscillator(
│   │   │   │   uid='osc_0',
│   │   │   │   frequency=212976765.0163832,
│   │   │   │   modulation_type=ModulationType.HARDWARE,
│   │   │   │   carrier_type=None
│   │   │   ),
│   │   │   port_delay=None,
│   │   │   port_mode=None,
│   │   │   range=-30,
│   │   │   threshold=None,
│   │   │   amplifier_pump=None,
│   │   │   added_outputs=[],
│   │   │   automute=True
│   │   ),
│   │   'measure': SignalCalibration(
│   │   │   amplitude=None,
│   │   │   delay_signal=None,
│   │   │   local_os

In [ ]:
qubit_params_module.sb_pulses

{'alice': {'f0g1': PulseFunctional(
│   function='gaussian_square_custom',
│   uid='sb_f0g1_alice_pulse',
│   amplitude=1,
│   length=3.1600000000000002e-06,
│   can_compress=True,
│   pulse_parameters={
│   │   'ramp': 1.6e-07
│   }
)
,
  'f1g2': PulseFunctional(
│   function='gaussian_square_custom',
│   uid='sb_f1g2_alice_pulse',
│   amplitude=0.6,
│   length=2.16e-06,
│   can_compress=True,
│   pulse_parameters={
│   │   'ramp': 1.6e-07
│   }
)
,
  'f2g3': PulseFunctional(
│   function='gaussian_square_custom',
│   uid='sb_f2g3_alice_pulse',
│   amplitude=1,
│   length=2.6600000000000004e-06,
│   can_compress=True,
│   pulse_parameters={
│   │   'ramp': 1.6e-07
│   }
)
,
  'f3g4': PulseFunctional(
│   function='gaussian_square_custom',
│   uid='sb_f3g4_alice_pulse',
│   amplitude=1,
│   length=2.6600000000000004e-06,
│   can_compress=True,
│   pulse_parameters={
│   │   'ramp': 1.6e-07
│   }
)
,
  'f4g5': PulseFunctional(
│   function='gaussian_square_custom',
│   uid='sb_f4g5_alic

In [ ]:
qubit_params_module.cav_alice

PulseFunctional(
│   function='const',
│   uid='cav_alice_pulse',
│   amplitude=0.1,
│   length=2e-05,
│   can_compress=True,
│   pulse_parameters=None
)
